# Session6. Кластеры для LightGBM

Контроль 0.326 на закрытом тесте использует две части. CatBoost с поправками задаёт общий объём горизонта, LightGBM уточняет его распределение по маршрутам. Здесь проверяем кластеризацию только для второй части. Признаки, параметры каждого горизонта, целевая переменная и веса берутся из Session6. Пять обучающих стартов в день, с 09:30 до 11:30, как в исходном решении. LightGBM обучается на CPU, как в исходном тренере.

В Kaggle подключите архив `best/cleaning_comparison_inputs.zip` как приватный input и выполните Run All. Если файлы уже распакованы, они должны лежать рядом с `comparison_meta.json`. Для локального запуска из репозитория используется тот же архив. Требуются NumPy, pandas, PyArrow, scikit-learn, joblib и LightGBM.

Обе ветки обучаются на сырой истории с целями строго до 4 октября. KMeans также видит только эту историю. Сравниваем общую модель и модели по 8 кластерам, небольшие кластеры используют общую модель. Срезы val и test включают прогнозы от 10:30 за 4–17 и 18–31 октября соответственно. Параметры и число кластеров фиксированы заранее, по test ничего не подбирается.

В основной таблице обе формы приведены к сумме новой общей LightGBM по каждому cutoff и горизонту. Поэтому разница WAPE показывает эффект распределения при одинаковом bias. Дополнительная таблица для test подставляет обе формы в архивный Session6 с его прежним весом и объёмом. Она сравнивает две заново обученные ветки в одном протоколе, а архивные 0.319190 служат отдельным ориентиром. Архивная ветка объёма использовала причинные поправки по доступным прошлым фактам. Локальный test уже изучался. Этот диагностический опыт сохраняет метрики и прогнозы и не собирает новый сабмит.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import gc
import json
import zipfile

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

work = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
search = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path.cwd() / 'best'
RUN = work / 'session6_lightgbm_clusters'
RUN.mkdir(exist_ok=True)
fit_end = pd.Timestamp('2025-10-04')
test_start = pd.Timestamp('2025-10-18')
test_end = pd.Timestamp('2025-11-01')
STEP = pd.Timedelta(minutes=30)
CLUSTERS = (1, 8)
MIN_CLUSTER_ROUTES = 10
SEED = 42
TARGET = 'target_1h'
names = [TARGET, *[f'status_{i}' for i in range(1, 7)]]
keys = ['cutoff', 'horizon', 'route_id']
group_keys = ['cutoff', 'horizon']
required = ['comparison_meta.json', 'train_solo_track.parquet', 'validation_predictions.parquet', 'evaluation_prediction_values.parquet']
temporary = TemporaryDirectory()
folders = [p.parent for p in search.rglob('comparison_meta.json') if all((p.parent / f).is_file() for f in required)]
if len(folders) == 1:
    data = folders[0]
else:
    archives = list(search.rglob('cleaning_comparison_inputs.zip'))
    if len(archives) != 1:
        raise ValueError('Подключите один архив cleaning_comparison_inputs.zip')
    data = Path(temporary.name)
    with zipfile.ZipFile(archives[0]) as archive:
        for filename in required:
            (data / filename).write_bytes(archive.read(filename))
meta = json.loads((data / 'comparison_meta.json').read_text())
history = pd.read_parquet(data / 'train_solo_track.parquet', columns=['timestamp', 'route_id', *names])
history = history.sort_values(['timestamp', 'route_id'])
times = pd.DatetimeIndex(history.timestamp.unique())
routes = np.sort(history.route_id.unique())
if len(history) != len(times) * len(routes) or history.duplicated(['timestamp', 'route_id']).any() or history.isna().any().any():
    raise ValueError('Неполная панель')
if not times.equals(pd.date_range(times[0], times[-1], freq='30min')):
    raise ValueError('Неверная сетка времени')
values = {name: history[name].to_numpy(dtype=np.float32).reshape(len(times), len(routes)) for name in names}
del history
target = values[TARGET]
TIMES, ROUTES, VALUES = times, routes, values


In [ ]:
def rolling(array, steps):
    return pd.DataFrame(array).rolling(steps, min_periods=1).mean().to_numpy(dtype=np.float32)

def extend_robust(data):
    new = {}
    floor = 1 / np.maximum(data.scale.to_numpy(dtype=float), 1)
    for j in range(1, 7):
        name = f'status_{j}'
        level = np.maximum(data[f'long_{name}_mean48'].to_numpy(dtype=float), floor)
        week = np.maximum(data[f'long_{name}_mean336'].to_numpy(dtype=float), floor)
        for suffix, key in [('current', f'{name}_lag0'), ('mean4', f'{name}_mean4'), ('mean12', f'{name}_mean12'), ('delta4', f'{name}_delta4')]:
            new[f'robust_s{j}_{suffix}'] = np.clip(data[key].to_numpy() / level, -100, 100).astype(np.float32)
        new[f'robust_s{j}_level_change'] = np.clip(level / week, 0, 100).astype(np.float32)
        raw = data[f'{name}_lag0'].to_numpy(dtype=float) * data.scale.to_numpy()
        denominator = pd.Series(raw).groupby([data.cutoff, data.horizon], sort=False).transform('sum').to_numpy()
        new[f'robust_s{j}_route_share'] = np.divide(raw, np.maximum(denominator, 1)).astype(np.float32)
    for window in [4, 12]:
        new[f'robust_target_vs{window}'] = np.clip(data.target_lag_0.to_numpy() / np.maximum(data[f'target_mean_{window}'].to_numpy(), floor), 0, 100).astype(np.float32)
    return pd.concat([data, pd.DataFrame(new, index=data.index)], axis=1)

def features(origins, h, profile_history, daily):
    T = len(times)
    R = len(routes)

    def past(a, index):
        index = np.asarray(index)
        out = a[np.clip(index, 0, T - 1)].astype(np.float32, copy=True)
        out[(index < 0) | (index >= T)] = np.nan
        return out
    scale = np.maximum(daily[origins], 1)
    common = {}
    for lag in (*range(9), 12, 24, 48, 96, 336):
        common[f'target_lag_{lag}'] = past(target, origins - lag) / scale
    for window in (4, 12, 48):
        common[f'target_mean_{window}'] = (daily if window == 48 else means['target_1h', window])[origins] / scale
    for window in (4, 12):
        common[f'target_std_{window}'] = deviations[window][origins] / scale
    for name in names[1:]:
        a = values[name]
        for lag in (0, 1, 2):
            common[f'{name}_lag{lag}'] = past(a, origins - lag) / scale
        for window in (4, 12):
            common[f'{name}_mean{window}'] = means[name, window][origins] / scale
        for lag in (2, 4):
            common[f'{name}_delta{lag}'] = (a[origins] - past(a, origins - lag)) / scale
    common['log_current_level'] = np.log1p(scale)
    for name, a in values.items():
        total = a.sum(axis=1, dtype=np.float64)
        netlevel = pd.Series(total).rolling(48, min_periods=1).mean().to_numpy()
        net12 = pd.Series(total).rolling(12, min_periods=1).mean().to_numpy()
        den = np.maximum(netlevel[origins], 1)
        for suffix, x in [('level', total[origins] / den), ('mean12', net12[origins] / den), ('delta2', (total[origins] - total[origins - 2]) / den)]:
            common[f'network_{name}_{suffix}'] = np.broadcast_to(x[:, None], scale.shape).astype(np.float32)
    common['route_network_share'] = target[origins] / np.maximum(target[origins].sum(axis=1, keepdims=True), 1)
    slot = origins + h
    dates = times[origins] + pd.Timedelta(minutes=30 * h)
    arrays = {name: a.ravel() for name, a in common.items()}
    for day in (1, 2, 3, 7, 14, 21, 28):
        a = past(target, slot - day * 48)
        arrays[f'slot_day{day}'] = (a / scale).ravel()
    for day in (1, 7):
        for offset in (-2, -1, 1, 2):
            arrays[f'slot_day{day}_offset{offset}'] = (past(target, slot - day * 48 + offset) / scale).ravel()
    stack = np.stack([past(target, slot - day * 48) for day in range(1, 8)])
    arrays['slot_daily7'] = (np.nanmean(stack, axis=0) / scale).ravel()
    stack = np.stack([past(target, slot - day * 48) for day in (7, 14, 21, 28)])
    with np.errstate(invalid='ignore'):
        weekly = np.nanmean(stack, axis=0)
    arrays['slot_weekly4'] = (weekly / scale).ravel()
    profile_sum = np.zeros_like(scale, dtype=np.float64)
    count = np.zeros_like(scale)
    for day in range(1, 29):
        a = past(profile_history, slot - day * 48) * scale / np.maximum(past(daily, origins - day * 48), 1)
        valid = np.isfinite(a)
        profile_sum += np.nan_to_num(a)
        count += valid
    profile = np.divide(profile_sum, count, out=np.zeros_like(profile_sum), where=count > 0).astype(np.float32)
    arrays['profile_norm'] = (profile / scale).ravel()
    arrays['horizon'] = np.full(len(origins) * R, h, dtype=np.int8)
    arrays['dayofweek'] = np.repeat(dates.dayofweek, R).astype(np.int8)
    arrays['origin_minute'] = np.repeat(times[origins].hour * 60 + times[origins].minute, R).astype(np.int16)
    arrays['future_hour'] = np.repeat(dates.hour + dates.minute / 60, R).astype(np.float32)
    frame = pd.DataFrame(arrays)
    frame['cutoff'] = times[origins].repeat(R)
    frame['timestamp'] = dates.repeat(R)
    frame['route_id'] = np.tile(routes, len(origins)).astype(np.int16)
    frame['scale'] = scale.ravel()
    frame['profile28'] = profile.ravel()
    frame['y'] = past(target, slot).ravel()
    extra = {}
    for name in names[1:]:
        for lag in (4, 8, 12, 24, 48, 96):
            extra[f'long_{name}_lag{lag}'] = (past(values[name], origins - lag) / scale).ravel()
        for window, series in [(48, means[name, 48]), (336, long_week[name])]:
            extra[f'long_{name}_mean{window}'] = (series[origins] / scale).ravel()
    return extend_robust(pd.concat([frame, pd.DataFrame(extra)], axis=1))

def metric(actual, predicted):
    y = np.asarray(actual, dtype=float)
    p = np.asarray(predicted, dtype=float)
    if y.shape != p.shape or not np.isfinite(y).all() or (not np.isfinite(p).all()) or (y.sum() <= 0):
        raise ValueError('Неверные данные для метрики')
    error = p - y
    wape = np.abs(error).sum() / y.sum()
    bias = error.sum() / y.sum()
    return dict(rows=len(y), WAPE=wape, relative_bias=bias, score=wape + abs(bias))

def segment_routes(boundary, stage):
    mask = TIMES < boundary
    y = VALUES[TARGET][mask].astype(np.float64)
    scale = np.maximum(y.mean(axis=0), 1.0)
    normalized = y / scale
    grid = TIMES[mask]
    descriptors = {}
    for slot in range(48):
        select = grid.hour * 2 + grid.minute // 30 == slot
        descriptors[f'slot_{slot:02}'] = normalized[select].mean(axis=0)
    for day in range(7):
        descriptors[f'weekday_{day}'] = normalized[grid.dayofweek == day].mean(axis=0)
    descriptors['log_cv'] = np.log1p(normalized.std(axis=0))
    descriptors['zero_share'] = (y == 0).mean(axis=0)
    descriptors['jump_share'] = (np.abs(np.diff(normalized, axis=0)) > 1).mean(axis=0)
    descriptors['log_q95'] = np.log1p(np.quantile(normalized, 0.95, axis=0))
    frame = pd.DataFrame(descriptors, index=ROUTES)
    if not np.isfinite(frame.to_numpy()).all():
        raise ValueError('Недостаточно данных для профилей маршрутов')
    scaler = StandardScaler().fit(frame)
    z = scaler.transform(frame)
    labels, estimators = ({1: np.zeros(len(ROUTES), dtype=int)}, {})
    for k in CLUSTERS:
        if k == 1:
            continue
        model = KMeans(n_clusters=k, n_init=20, random_state=SEED).fit(z)
        groups = model.labels_.copy()
        counts = np.bincount(groups, minlength=k)
        groups[counts[groups] < MIN_CLUSTER_ROUTES] = -1
        labels[k], estimators[k] = (groups, model)
    artifact = dict(boundary=str(boundary), routes=ROUTES, scale=scale, labels=labels, descriptor_scaler=scaler, clusterers=estimators, descriptor_columns=list(frame.columns))
    joblib.dump(artifact, RUN / f'segments_{stage}.joblib')
    frame.insert(0, 'route_id', ROUTES)
    frame['target_scale'] = scale
    for k, groups in labels.items():
        frame[f'cluster_{k}'] = groups
    frame.to_csv(RUN / f'route_profiles_{stage}.csv', index=False)
    display(pd.DataFrame([dict(stage=stage, k=k, cluster=int(g), routes=int((groups == g).sum())) for k, groups in labels.items() for g in np.unique(groups)]))
    return artifact

In [ ]:
means = {(name, steps): rolling(array, steps) for name, array in values.items() for steps in (4, 12, 48)}
level24 = means[TARGET, 48]
long_week = {name: rolling(values[name], 336) for name in names[1:]}
deviations = {w: pd.DataFrame(target).rolling(w, min_periods=2).std().to_numpy(dtype=np.float32) for w in (4, 12)}
with threadpool_limits(limits=4):
    segmentation = segment_routes(fit_end, 'before_val')
route_cluster = pd.Series(segmentation['labels'][8], index=routes)
positions = np.arange(len(times))
minutes = times.hour * 60 + times.minute
train_starts = np.isin(minutes, [570, 600, 630, 660, 690]) & (positions >= 336)
query_starts = (times >= fit_end) & (times < test_end) & (minutes == 630)
fit_reports = []
predictions = []

for h in range(1, 9):
    train_origins = positions[train_starts & (times + h * STEP < fit_end)]
    query_origins = positions[query_starts & (positions + h < len(times))]
    train = features(train_origins, h, target, level24)
    query = features(query_origins, h, target, level24)
    config = meta[str(h)]
    columns = config['columns']
    if set(columns) & {'y', 'cutoff', 'timestamp'}:
        raise ValueError('Недопустимые колонки LightGBM')
    train_scale = train.scale.to_numpy(dtype=float)
    query_scale = query.scale.to_numpy(dtype=float)
    train_group = train.route_id.map(route_cluster).to_numpy()
    query_group = query.route_id.map(route_cluster).to_numpy()
    query_period = np.where(query.cutoff < test_start, 'val', 'test')
    params = dict(config['params'], n_estimators=config['trees'], n_jobs=4)
    global_pred = np.empty(len(query))
    clustered_pred = np.empty(len(query))
    groups = ['global', *sorted(set(train_group) - {-1})]
    for group in groups:
        train_mask = np.ones(len(train), dtype=bool) if group == 'global' else train_group == group
        query_mask = np.ones(len(query), dtype=bool) if group == 'global' else query_group == group
        model = lgb.LGBMRegressor(**params)
        model.fit(train.loc[train_mask, columns], train.y.to_numpy()[train_mask] / train_scale[train_mask],
                  sample_weight=train_scale[train_mask] / train_scale.mean())
        model.booster_.save_model(str(RUN / f'lightgbm_h{h}_{group}.txt'))
        train_pred = np.maximum(model.predict(train.loc[train_mask, columns]) * train_scale[train_mask], 0)
        query_pred = np.maximum(model.predict(query.loc[query_mask, columns]) * query_scale[query_mask], 0)
        if group == 'global':
            global_pred[:] = query_pred
            clustered_pred[:] = query_pred
        else:
            clustered_pred[query_mask] = query_pred
        rows = [dict(h=h, group=group, period='train', **metric(train.y.to_numpy()[train_mask], train_pred))]
        for period in ('val', 'test'):
            mask = query_period[query_mask] == period
            rows.append(dict(h=h, group=group, period=period, **metric(query.y.to_numpy()[query_mask][mask], query_pred[mask])))
        fit_reports.extend(rows)
        display(pd.DataFrame(rows))
        pd.DataFrame(fit_reports).to_csv(RUN / 'metrics_after_fit.csv', index=False)
        del model, train_pred, query_pred
    predictions.append(query[keys + ['timestamp', 'y']].assign(period=query_period, cluster=query_group,
        global_shape=global_pred, clustered_shape=clustered_pred))
    del train, query
    gc.collect()


In [ ]:
def normalize_shape(frame, shape, anchor):
    amounts = frame[group_keys + [shape, anchor]].copy()
    amounts[[shape, anchor]] = amounts[[shape, anchor]].astype(float)
    shape_total = amounts.groupby(group_keys, sort=False)[shape].transform('sum').to_numpy()
    anchor_total = amounts.groupby(group_keys, sort=False)[anchor].transform('sum').to_numpy()
    result = amounts[anchor].to_numpy().copy()
    valid = shape_total > 0
    result[valid] = amounts[shape].to_numpy()[valid] * anchor_total[valid] / shape_total[valid]
    totals = amounts.assign(prediction=result).groupby(group_keys, sort=False)[['prediction', anchor]].sum()
    if not np.allclose(totals.prediction, totals[anchor], rtol=1e-10, atol=1e-5):
        raise ValueError('Не сохранён общий объём')
    return result


frame = pd.concat(predictions, ignore_index=True)
if frame.duplicated(keys).any() or len(frame) != 28 * 8 * len(routes):
    raise ValueError('Неполная панель прогнозов')
frame['clustered_at_global_volume'] = normalize_shape(frame, 'clustered_shape', 'global_shape')
frame.to_parquet(RUN / 'shape_predictions.parquet', index=False)
rows = []
for period, part in frame.groupby('period'):
    for column in ('global_shape', 'clustered_shape', 'clustered_at_global_volume'):
        rows.append(dict(protocol='shape', period=period, variant=column, **metric(part.y, part[column])))

archive_test = pd.read_parquet(data / 'validation_predictions.parquet')
volume = pd.read_parquet(data / 'evaluation_prediction_values.parquet')[keys + ['calibrated_prediction']]
test = frame.loc[frame.period.eq('test')].merge(archive_test[keys + ['y', 'session_prediction']], on=keys,
    validate='one_to_one', how='left', suffixes=('', '_archive')).merge(volume, on=keys, validate='one_to_one', how='left')
if not np.array_equal(test.y, test.y_archive) or test.calibrated_prediction.isna().any():
    raise ValueError('Архивный test не совпадает с оценкой')
weight = float(meta['weight'])
for shape in ('global_shape', 'clustered_shape'):
    test[f'session_{shape}'] = (1 - weight) * test.calibrated_prediction + weight * normalize_shape(test, shape, 'calibrated_prediction')
session_columns = ['session_prediction', 'session_global_shape', 'session_clustered_shape']
for column in session_columns:
    rows.append(dict(protocol='archived_volume', period='test', variant=column, **metric(test.y, test[column])))
    sums = test.groupby(group_keys)[[column, 'calibrated_prediction']].sum()
    if not np.allclose(sums[column], sums.calibrated_prediction, rtol=1e-10, atol=1e-5):
        raise ValueError('Объём Session6 изменился')
scores = pd.DataFrame(rows)
display(scores)
scores.to_csv(RUN / 'scores.csv', index=False)
test.to_parquet(RUN / 'session6_volume_test_predictions.parquet', index=False)
for key in ('cutoff', 'horizon', 'cluster'):
    details = [dict(**{key: value}, variant=column, **metric(part.y, part[column]))
               for value, part in test.groupby(key) for column in session_columns]
    pd.DataFrame(details).to_csv(RUN / f'test_by_{key}.csv', index=False)
(RUN / 'protocol.json').write_text(json.dumps(dict(fit_end_exclusive=str(fit_end), test_start=str(test_start),
    test_end_exclusive=str(test_end), clusters=list(CLUSTERS), min_cluster_routes=MIN_CLUSTER_ROUTES,
    shape_weight=weight, test_used_for_selection=False, evaluation_before_full_refit=True, full_refit=False,
    parameters=meta), ensure_ascii=False, indent=2))
with zipfile.ZipFile(RUN.with_suffix('.zip'), 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN.iterdir()):
        if path.is_file():
            archive.write(path, path.name)
print('Локальный test оценён до полного переобучения. Период ранее изучался.')
print('Архив результатов', RUN.with_suffix('.zip'))
